In [1]:
import os
import time
from dotenv import load_dotenv

load_dotenv()

print("=== Environment Check ===")
print(f"GOOGLE_API_KEY set: {bool(os.environ.get('GOOGLE_API_KEY'))}")
print(f"GROQ_API_KEY set:   {bool(os.environ.get('GROQ_API_KEY'))}")

=== Environment Check ===
GOOGLE_API_KEY set: True
GROQ_API_KEY set:   True


In [8]:
print("\n=== Gemini Test ===\n")

try:
    from google import genai

    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

    # Simple generate_content call
    start = time.time()
    for i in range(20):
        resp = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents="""
            A 36-year-old man is brought to the emergency department by his wife 20 minutes after having a seizure. 
            Over the past 3 days, he has had a fever and worsening headaches. 
            This morning, his wife noticed that he was irritable and demonstrated strange behavior; he put the back of his fork, the salt shaker, and the lid of the coffee can into his mouth. 
            He has no history of serious illness and takes no medications. His temperature is 39\u00b0C (102.2\u00b0F), pulse is 88/min, and blood pressure is 118/76 mm Hg. 
            Neurologic examination shows diffuse hyperreflexia and an extensor response to the plantar reflex on the right. 
            A T2-weighted MRI of the brain shows edema and areas of hemorrhage in the left temporal lobe. Which of the following is most likely the primary mechanism of the development of edema in this patient?, "option_a": "Release of vascular endothelial growth factor", "option_b": "Cellular retention of sodium", "option_c": "Breakdown of endothelial tight junctions", "option_d": "Increased hydrostatic pressure
            """,
        config={
            "temperature": 0.0,
            "max_output_tokens": 64,
            },
        )
        latency = (time.time() - start) * 1000

        print(f"Response: {resp.text}")
        print(f"Latency:  {latency:.0f} ms")
        print("✅ Gemini is working\n")

except Exception as e:
    print(f"❌ Gemini failed: {type(e).__name__}: {e}\n")


=== Gemini Test ===

Response: The correct answer is **"option_c": "Breakdown of endothelial tight junctions"**. Here's why:

*   **The Patient's Presentation:** The patient has a history of fever, worsening headaches, irritability, strange behavior (suggesting altered mental status and potentially focal neurological deficits), and a seizure
Latency:  1061 ms
✅ Gemini is working

Response: The correct answer is **"option_c": "Breakdown of endothelial tight junctions"**. Here's why:

*   **The Patient's Presentation:** The patient has a history of fever, worsening headaches, irritability, strange behavior (suggesting altered mental status and potentially focal neurological deficits), and a seizure
Latency:  1854 ms
✅ Gemini is working

Response: The correct answer is **"option_c": "Breakdown of endothelial tight junctions"**. Here's why:

*   **The Patient's Presentation:** The patient has a history of fever, worsening headaches, irritability, strange behavior (suggesting altered menta

KeyboardInterrupt: 

In [7]:
print("=== Groq Test ===\n")

try:
    from groq import Groq

    client = Groq(api_key=os.environ["GROQ_API_KEY"])

    start = time.time()
    for i in range(1):
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": """
            A 36-year-old man is brought to the emergency department by his wife 20 minutes after having a seizure. 
            Over the past 3 days, he has had a fever and worsening headaches. 
            This morning, his wife noticed that he was irritable and demonstrated strange behavior; he put the back of his fork, the salt shaker, and the lid of the coffee can into his mouth. 
            He has no history of serious illness and takes no medications. His temperature is 39\u00b0C (102.2\u00b0F), pulse is 88/min, and blood pressure is 118/76 mm Hg. 
            Neurologic examination shows diffuse hyperreflexia and an extensor response to the plantar reflex on the right. 
            A T2-weighted MRI of the brain shows edema and areas of hemorrhage in the left temporal lobe. Which of the following is most likely the primary mechanism of the development of edema in this patient?, "option_a": "Release of vascular endothelial growth factor", "option_b": "Cellular retention of sodium", "option_c": "Breakdown of endothelial tight junctions", "option_d": "Increased hydrostatic pressure"""
            }],
            temperature=0.0,
            max_tokens=64,
        )
        latency = (time.time() - start) * 1000

        print(f"Response: {resp.choices[0].message.content}")
        print(f"Latency:  {latency:.0f} ms")
    print("✅ Groq is working\n")

except Exception as e:
    print(f"❌ Groq failed: {type(e).__name__}: {e}\n")

=== Groq Test ===

Response: The primary mechanism of the development of edema in this patient is most likely "option_b": "Cellular retention of sodium".

The patient's symptoms and MRI findings suggest a diagnosis of herpes simplex encephalitis (HSE), which is a viral infection of the brain caused by the herpes simplex virus (HSV).
Latency:  405 ms
✅ Groq is working



In [ ]:
print("=== Testing via call_model() ===\n")

# Adjust this import to wherever your module lives
try:
    from llm_client import call_model  # or whatever your file is named
except ImportError:
    # Inline fallback — paste call_model here or fix the import path
    print("⚠️  Could not import call_model. Skipping wrapper test.")
    call_model = None

if call_model:
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 2 + 2? Answer in one word."},
    ]

    for provider, model in [
        ("google", "gemini-2.0-flash"),
        ("groq", "llama-3.1-8b-instant"),
    ]:
        try:
            result = call_model(
                provider=provider,
                model=model,
                messages=test_messages,
                temperature=0.0,
                max_tokens=32,
            )
            print(f"{provider}/{model}")
            print(f"  Response: {result['text']}")
            print(f"  Latency:  {result['latency_ms']:.0f} ms")
            print(f"  ✅ OK\n")
        except Exception as e:
            print(f"{provider}/{model}")
            print(f"  ❌ {type(e).__name__}: {e}\n")

In [ ]:
print("=== Available Gemini Models ===\n")

try:
    from google import genai

    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    for m in client.models.list():
        if "gemini" in m.name.lower():
            print(f"  {m.name}")
except Exception as e:
    print(f"Could not list models: {e}")